In [ ]:
import os

import pandas as pd
import yaml

from pynxtools_apm import get_pynxtools_apm_version
from pynxtools_apm.examples.deu_berlin_kuehbach.batch import process_project
from pynxtools_apm.examples.deu_berlin_kuehbach.utils import get_project_id

with open(f"{os.getcwd()}{os.sep}/config.yaml", encoding="utf-8") as fp:
    config: dict[str, str] = yaml.safe_load(fp)
for task, location in config.items():
    print(f"{task}, {location}")

# os.makedirs(f"{trg_directory.replace('/decompressed', '/pynxtools')}", exist_ok=True)
# os.listdir(f"{trg_directory.replace('/decompressed', '')}")

In [ ]:
spread_sheet_of_all_projects = pd.read_excel(
    f"{config['data']}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype=str,
).fillna("")

project_range: tuple[int, int] = (1, 160)

with open(f"{config['data']}{os.sep}aaa_apm_nomad_project_names.yaml") as fp:
    nomad_project_names: dict[str, str] = yaml.safe_load(fp)

In [ ]:
count: int = 0
cli_script: list[str] = [
    "#!/bin/bash",
    f"# autogenerated by pynxtools-apm v{get_pynxtools_apm_version()}",
    "",
]

for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        project_id = get_project_id(row.id)
        if project_range[0] <= int(project_id) <= project_range[1]:
            # if row.legal == "1" and row.use == "1":
            if project_id in nomad_project_names:
                if project_id not in ("073"):
                    continue
                print(f"Adding {row.id} {row.project_name}")
                cli_script.append(f"date -Iseconds")
                cli_script.append(f"echo '{project_id}'")
                cli_script.append(
                    f"python3 04_batch_process.py --project='{project_id}'"
                )
                # break

cli_script.append(f"date -Iseconds")
cli_script.append("echo 'Batch queue completed'")
cli_script_file_path = f"{os.getcwd()}{os.sep}04_batch_process.sh"
with open(cli_script_file_path, "w", encoding="utf-8") as fp:
    fp.write("\n".join(cli_script))
os.chmod(cli_script_file_path, 0o755)

***

All code below is considered deprecated because Jupyter notebook memory management and stream forwarding causes unnecessary overhead over pure Python process.

## Deprecated

In [ ]:
import os

import pandas as pd
import yaml

from pynxtools_apm import SEPARATOR
from pynxtools_apm.examples.deu_berlin_kuehbach.batch import process_project
from pynxtools_apm.examples.deu_berlin_kuehbach.utils import (
    get_project_id,
    snake_case_to_camel_case,
)

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory: str = f"{fp.read().strip().replace('/', os.sep)}"
print(src_directory)
with open("target_directory.txt") as fp:
    trg_directory: str = f"{fp.read().strip().replace('/', os.sep)}"
print(trg_directory)

os.makedirs(f"{trg_directory.replace('/decompressed', '/pynxtools')}", exist_ok=True)
os.listdir(f"{trg_directory.replace('/decompressed', '')}")

In [ ]:
spread_sheet_of_all_projects = pd.read_excel(
    f"{src_directory}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype=str,
).fillna("")

with open(f"{src_directory}{os.sep}aaa_apm_nomad_project_names.yaml") as fp:
    nomad_project_names: dict[str, str] = yaml.safe_load(fp)

project_range: tuple[int, int] = (73, 73)
# TODO fix issues for project_id
# 36, some assignments broken
# 113 (too large allocations, likely unphysically large x, y, z coordinates)
# 149 some assignments broken
# bilal, (6, 6)
# pauly, (73, 73)
# uzuhashi (101, 101)

In [ ]:
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        if project_range[0] <= int(row.id) <= project_range[1]:
            project_id = get_project_id(row.id)

            print(
                f"project{SEPARATOR}{row.project_name}{SEPARATOR}{project_id}{SEPARATOR}batch..."
            )

            if project_id in nomad_project_names:
                process_project(
                    project_id,
                    row.project_name,
                    f"{src_directory}{os.sep}{row.project_name}.ods",
                    f"{src_directory}{os.sep}aaa_legacy_data.bib",
                    f"{src_directory}{os.sep}{row.project_name}.sha256.results.csv",
                    trg_directory,
                    f"{trg_directory.replace('/decompressed', '/pynxtools')}",
                    openalex_file=f"{os.getcwd()}{os.sep}openalex/D_{snake_case_to_camel_case(row.project_name)}.json",
                    logger_file_path_suffix="paper.run01",
                    nomad_project_name=nomad_project_names[project_id],
                )
print("Batch queue completed")